以上定义输出结构的四种模式中，我们都是通过调用“with_structured_output”来获取结构化输出结
果，除了这种方式外，还可以通过使用输出解释器来获取结构化输出结果。下面介绍这两种获取结构化
结果的方式。
4.1 使用with_structured_output
这种方式是
最新、
最简洁的API，直接让模型“理解”你需要的数据结构，并返回解析好的对象。
此外，我们可以在with_structured_output方法中传入
include_raw=True 参数，表示返回解析前的
始AIMessage ，从而访问令牌用量等元数据。

In [15]:
from dotenv import load_dotenv
import os
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}  #关闭思考模式
)

In [16]:
from pydantic import BaseModel, Field
from rich import print as rprint


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分（10分制）")


# 设置模型结构化输出
model_with_structure = model.with_structured_output(Movie, include_raw=True)
# 调用模型并获取结构化输出
resp = model_with_structure.invoke("给我介绍下电影《星际穿越》")
print(type(resp))
rprint(resp)

<class 'dict'>


{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 36,
                'prompt_tokens': 351,
                'total_tokens': 387,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                'prompt_cache_hit_tokens': 256,
                'prompt_cache_miss_tokens': 95
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
            'id': '2972f904-1123-42ae-8af2-bb27af66ded6',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f60a0-d85b-7b93-92cf-e9fdbe1e98ad-0',
        tool_calls=[
            {
                'name': 'Movie',
                'args': {'title': '星际穿越'},
                'id': 'call_00_SVuaFybrP2pUlezhRG5h7795',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 351,
            'output_tokens': 36,
            'total_tokens': 387,
            'input_token_details': {'cache_read': 256},
            'output_token_details': {}
        }
    ),
    'parsing_error': 3 validation errors for Movie
year
  Field required [type=missing, input_value={'title': '星际穿越'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
director
  Field required [type=missing, input_value={'title': '星际穿越'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
rating
  Field required [type=missing, input_value={'title': '星际穿越'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing,
    'parsed': None
}

输出包含了完整的输出响应，包含三个字段
raw：返回的原始AIMessage。
parsed：解析后的输出
parsing_error：解析错误，当前用的是Pydantic，校验，格式不符合schema会导致报错。其它
三种方式不符合schema不会导致报错。

4.2
使用输出解析器(不推荐) langchain0.3版本的方法
这种方法更传统，依赖于在提示词中明确指示模型输出特定格式的文本，然后使用解析器进行转换。
其流程是：提示词指导(引导生成指定类型）→ 模型生成文本 → 解析器转换。

In [19]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


# 1. 创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "回答用户问题,并根据用户输入提取信息。\n{schema}"),
    ("human", "问题：{question}")
])

# 3. 定义结构
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)
#生成提示词对应的模板
print(parser.get_format_instructions())
# 5. 创建链
chain = prompt_template | model | parser
# 6. 调用（返回字典）  将用户问题 和提示词模板注入到链中，模型会根据提示词模板生成文本，解析器会将文本转换为结构化数据
response = chain.invoke({"question": "介绍电影《西红柿首富》", "schema": parser.get_format_instructions()})
#
# response = parser.invoke(model.invoke(prompt_template.invoke({"question": "介绍电影《盗梦空间》"})))
print(response)

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [20]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


# 1. 创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("human", "问题：{question}")
])

# 3. 定义结构
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)
# 5. 创建链
chain = prompt_template | model | parser
# 6. 调用（返回字典）
response = chain.invoke({"question": "介绍电影《西红柿首富》"})
#
# response = parser.invoke(model.invoke(prompt_template.invoke({"question": "介绍电影《盗梦空间》"})))
print(response)

OutputParserException: Invalid json output: 《西红柿首富》是一部2018年上映的中国喜剧电影，由闫非、彭大魔执导，沈腾、宋芸桦、张一鸣、常远等主演。影片改编自1985年的美国电影《布鲁斯特的百万横财》，讲述了落魄守门员王多鱼（沈腾 饰）意外获得继承二爷巨额遗产的机会，但必须在规定时间内花光十亿人民币的故事。

影片中，王多鱼原本生活困顿，却在此时得知自己有一位神秘的二爷留下三百亿遗产，但继承条件是在一个月内花光十亿，且不能捐赠、不能违法，也不能购买固定资产。王多鱼因此展开了一系列荒唐而疯狂的“花钱大作战”，包括投资烂尾楼、开发奇葩保险、举办“脂肪险”、聘请股神巴菲特共进午餐等。然而，他越是努力花钱，反而越是意外赚钱，引发了一系列令人捧腹的笑料。

电影在喜剧的外壳下，探讨了金钱与人性的关系，讽刺了资本对人性的扭曲与异化，同时也传递出“金钱并非万能”的主题。影片中沈腾的表演风格鲜明，夸张中带着真实，让观众在欢笑之余也能有所思考。

《西红柿首富》凭借其幽默的剧情、出色的表演和深刻的社会寓意，在国内取得了超过25亿人民币的票房成绩，成为当年国产喜剧电影中的一匹黑马。除此之外，影片中的“发明大王”陆游器（吐槽网络游戏中的“猪队友”）、王多鱼赤身追赶医生等桥段也成为了观众津津乐道的经典笑点。

总体而言，这是一部轻松搞笑又不乏深度的商业喜剧片，适合在放松心情时观看。
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 